# TSVD zadanie - spracovanie velkych dat v Apache Spark

Cielom je spojit merania prudov a napati s evidenciou poruch, pripravit denne priebehy a natrenovat klasifikacne modely.

V notebooku su ponechane vsetky povinne casti zadania:

- integracia dat a riesenie sucasnych poruch,
- stratifikovana 10 % vzorka,
- chybajuce hodnoty, outliery a popisne charakteristiky,
- denne priebehy s agregaciami,
- train/test rozdelenie,
- korelacie a vyber atributov,
- 3 binarne modely a 3 multiclass modely,
- kontingencna tabulka, precision, recall, F1 a MCC.

## 1. Importy a nastavenia

Notebook predpoklada, ze vstupne subory su v `C:\TSVD_zadanie\dataset`.

In [1]:
from pathlib import Path
import os
import math

import jdk4py
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier, DecisionTreeClassifier
from pyspark.ml.feature import Imputer, StringIndexer, VectorAssembler
from pyspark.ml.stat import Correlation

PROJECT_DIR = Path(r"C:\TSVD_zadanie")
DATASET_DIR = PROJECT_DIR / "dataset"
PREPARED_DIR = PROJECT_DIR / "prepared"

RAW_PARQUET = DATASET_DIR / "priebehy.parquet"
FAULTS_XLSX = DATASET_DIR / "Poruchy.xlsx"
SPARK_PARQUET = PREPARED_DIR / "priebehy_spark.parquet"

NUMERIC_COLS = ["u1_norm", "u2_norm", "u3_norm", "i1_norm", "i2_norm", "i3_norm"]
SEED = 42

## 2. Priprava Parquet suboru

Povodny Parquet ma cas `t_utc` ulozeny v nanosekundach. Spark 3.5 ma s tym na Windowse problem, preto ho raz preulozim na mikrosekundy. Hodnoty merani tym nemenim.

In [2]:
if not SPARK_PARQUET.exists():
    PREPARED_DIR.mkdir(exist_ok=True)
    parquet_file = pq.ParquetFile(RAW_PARQUET)
    writer = None

    for i in range(parquet_file.num_row_groups):
        table = parquet_file.read_row_group(i)

        new_fields = []
        for field in table.schema:
            if field.name == "t_utc":
                new_fields.append(pa.field("t_utc", pa.timestamp("us", tz="UTC")))
            else:
                new_fields.append(field)

        table = table.cast(pa.schema(new_fields))

        if writer is None:
            writer = pq.ParquetWriter(SPARK_PARQUET, table.schema, compression="snappy")
        writer.write_table(table)

    writer.close()

print(SPARK_PARQUET)

C:\TSVD_zadanie\prepared\priebehy_spark.parquet


## 3. Spustenie Spark session

In [3]:
os.environ["JAVA_HOME"] = str(jdk4py.JAVA_HOME)
os.environ["PYSPARK_PYTHON"] = str(PROJECT_DIR / ".venv" / "Scripts" / "python.exe")
os.environ["PYSPARK_DRIVER_PYTHON"] = os.environ["PYSPARK_PYTHON"]

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("TSVD zadanie")
    .config("spark.driver.memory", "6g")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "32")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

## 4. Nacitanie merani a poruch

In [4]:
measurements = (
    spark.read.parquet(str(SPARK_PARQUET))
    .drop("elektromer")
    .withColumn("t_utc", F.col("t_utc").cast("timestamp"))
    .cache()
)

print("Pocet merani:", measurements.count())
measurements.printSchema()
measurements.show(5, truncate=False)

Pocet merani: 7659371
root
 |-- t_utc: timestamp (nullable = true)
 |-- u2_norm: double (nullable = true)
 |-- i3_norm: double (nullable = true)
 |-- i2_norm: double (nullable = true)
 |-- i1_norm: double (nullable = true)
 |-- u1_norm: double (nullable = true)
 |-- u3_norm: double (nullable = true)
 |-- eic: string (nullable = true)

+-------------------+-------+-------+-------+-------+-------+-------+----------------+
|t_utc              |u2_norm|i3_norm|i2_norm|i1_norm|u1_norm|u3_norm|eic             |
+-------------------+-------+-------+-------+-------+-------+-------+----------------+
|2022-12-16 23:10:00|14123.0|0.0    |0.0    |0.0    |12703.0|12814.0|24ZVS0000003196R|
|2022-12-16 23:20:00|14068.0|0.0    |0.0    |0.0    |12658.0|12747.0|24ZVS0000003196R|
|2022-12-16 23:30:00|14087.0|0.0    |0.0    |0.0    |12634.0|12758.0|24ZVS0000003196R|
|2022-12-16 23:40:00|14179.0|0.0    |0.0    |0.0    |12632.0|12863.0|24ZVS0000003196R|
|2022-12-16 23:50:00|14174.0|0.0    |0.0    |0.0    |1

In [5]:
faults_pd = pd.read_excel(FAULTS_XLSX, sheet_name="Evidencia poruch")
fault_types_pd = pd.read_excel(FAULTS_XLSX, sheet_name="Typy poruch")

# Premenovanie podla poradia stlpcov, aby notebook nebol citlivy na diakritiku v Exceli.
faults_pd = faults_pd.rename(columns={
    faults_pd.columns[0]: "eic",
    faults_pd.columns[1]: "fault_start",
    faults_pd.columns[2]: "fault_end",
    faults_pd.columns[3]: "fault_type",
})

faults_pd = faults_pd.dropna(subset=["eic", "fault_type"])
faults_pd["fault_start"] = pd.to_datetime(faults_pd["fault_start"])
faults_pd["fault_end"] = pd.to_datetime(faults_pd["fault_end"])
faults_pd["fault_type"] = faults_pd["fault_type"].astype(int)

faults = spark.createDataFrame(faults_pd[["eic", "fault_start", "fault_end", "fault_type"]]).cache()

print("Pocet zaznamov poruch:", faults.count())
faults.show(10, truncate=False)
fault_types_pd

Pocet zaznamov poruch: 106
+----------------+-------------------+-------------------+----------+
|eic             |fault_start        |fault_end          |fault_type|
+----------------+-------------------+-------------------+----------+
|24ZVS0000003196R|2023-01-16 23:00:00|2023-06-06 22:00:00|4         |
|24ZVS0000001976B|2022-12-22 23:00:00|2023-03-13 23:00:00|5         |
|24ZVS0000002341C|2023-01-25 23:00:00|2024-07-24 22:00:00|4         |
|24ZVS0000003211K|NULL               |2023-10-22 22:00:00|3         |
|24ZVS0000639518F|2023-03-18 23:00:00|2024-08-21 22:00:00|3         |
|24ZVS0000038467G|NULL               |2023-06-26 22:00:00|1         |
|24ZVS0000074609I|NULL               |2023-06-23 22:00:00|1         |
|24ZVS0000002475S|2023-09-04 22:00:00|2023-09-07 22:00:00|4         |
|24ZVS00000020943|2023-10-05 22:00:00|2023-11-05 23:00:00|4         |
|24ZVS0000018647Q|2023-07-24 22:00:00|2023-11-21 23:00:00|4         |
+----------------+-------------------+-------------------+-----

,Číselník,Popis poruchy
0,1,Chybný PTP vo fáze L1
1,2,Chybný PTP vo fáze L2
2,3,Chybný PTP vo fáze L3
3,4,Výpadok napätia vo fáze L1
4,5,Výpadok napätia vo fáze L2
5,6,Výpadok napätia vo fáze L3
6,7,Porucha P34 vo fáze L1
7,8,Porucha P34 vo fáze L2
8,9,Porucha P34 vo fáze L3
9,13,Vykratený prúd vo fáze L1


## 5. Integracia dat

Porucha je interval od zaciatku do konca. Meranie dostane poruchu vtedy, ked ma rovnake `eic` a cas merania patri do intervalu poruchy.

Ak je v jednom case viac poruch naraz, ulozim ich ako mnozinu typov. Napriklad kombinacia `1+4` znamena, ze naraz platili typy poruch 1 a 4.

In [6]:
time_bounds = measurements.agg(F.min("t_utc").alias("min_time"), F.max("t_utc").alias("max_time"))

fault_intervals = (
    faults.crossJoin(time_bounds)
    .withColumn("start_time", F.coalesce(F.col("fault_start"), F.col("min_time")))
    .withColumn("end_time", F.coalesce(F.to_timestamp(F.date_add(F.to_date("fault_end"), 1)), F.col("max_time") + F.expr("INTERVAL 1 SECOND")))
    .select("eic", "start_time", "end_time", "fault_type")
)

m = measurements.withColumn("row_id", F.monotonically_increasing_id()).alias("m")
f = fault_intervals.alias("f")

joined = m.join(
    F.broadcast(f),
    (F.col("m.eic") == F.col("f.eic")) &
    (F.col("m.t_utc") >= F.col("f.start_time")) &
    (F.col("m.t_utc") < F.col("f.end_time")),
    "left"
)

labels = (
    joined.groupBy("row_id")
    .agg(
        F.array_sort(F.collect_set("fault_type")).alias("fault_types"),
        F.max(F.when(F.col("fault_type").isNotNull(), 1).otherwise(0)).alias("target_binary")
    )
    .withColumn("fault_types", F.expr("filter(fault_types, x -> x is not null)"))
    .withColumn("target_multiclass", F.when(F.size("fault_types") == 0, "0").otherwise(F.concat_ws("+", F.col("fault_types"))))
)

integrated = m.join(labels, "row_id").drop("row_id").cache()

integrated.groupBy("target_binary", "target_multiclass").count().orderBy("target_binary", "target_multiclass").show(50, truncate=False)

+-------------+-----------------+-------+
|target_binary|target_multiclass|count  |
+-------------+-----------------+-------+
|0            |0                |3233043|
|1            |1                |1554590|
|1            |13               |32850  |
|1            |2                |586212 |
|1            |3                |1323365|
|1            |4                |266708 |
|1            |5                |94620  |
|1            |6                |68899  |
|1            |7                |222251 |
|1            |8                |194591 |
|1            |9                |82242  |
+-------------+-----------------+-------+



## 6. Stratifikovana 10 % vzorka

In [7]:
fractions = {row["target_binary"]: 0.10 for row in integrated.select("target_binary").distinct().collect()}
sample_10 = integrated.sampleBy("target_binary", fractions=fractions, seed=SEED).cache()

print("Pocet riadkov v 10 % vzorke:", sample_10.count())
sample_10.groupBy("target_binary").count().orderBy("target_binary").show()

Pocet riadkov v 10 % vzorke: 764928
+-------------+------+
|target_binary| count|
+-------------+------+
|            0|322457|
|            1|442471|
+-------------+------+



## 7. Chybajuce hodnoty a popisne charakteristiky

In [8]:
missing_values = integrated.agg(*[
    F.count(F.when(F.col(c).isNull() | F.isnan(c), c)).alias(c)
    for c in NUMERIC_COLS
])
missing_values.show()

integrated.select(NUMERIC_COLS).summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max").show(truncate=False)

integrated.agg(
    *[F.skewness(c).alias(c + "_skewness") for c in NUMERIC_COLS],
    *[F.kurtosis(c).alias(c + "_kurtosis") for c in NUMERIC_COLS]
).show(truncate=False)

+-------+-------+-------+-------+-------+-------+
|u1_norm|u2_norm|u3_norm|i1_norm|i2_norm|i3_norm|
+-------+-------+-------+-------+-------+-------+
|  77638|  77622|  77638|  77638|  77622|  77638|
+-------+-------+-------+-------+-------+-------+

+-------+------------------+-----------------+------------------+------------------+------------------+------------------+
|summary|u1_norm           |u2_norm          |u3_norm           |i1_norm           |i2_norm           |i3_norm           |
+-------+------------------+-----------------+------------------+------------------+------------------+------------------+
|count  |7581733           |7581749          |7581733           |7581733           |7581749           |7581733           |
|mean   |736.7045705509078 |738.5095031924687|763.7825256472858 |11.359328966246457|11.923238116111174|12.04502518686281 |
|stddev |2561.4779262403267|2505.182530877948|2538.8969524941244|26.63775450154358 |28.21802047901344 |28.402429391343123|
|min    |0.

## 8. Outliery a nahradenie chybajucich hodnot

In [9]:
# Outliery riesim jednoducho cez IQR: hodnoty mimo hranic sa orezu na hranicu.
cleaned = integrated
outlier_rows = []

for c in NUMERIC_COLS:
    q1, q3 = integrated.approxQuantile(c, [0.25, 0.75], 0.01)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier_rows.append((c, q1, q3, lower, upper))

    cleaned = cleaned.withColumn(
        c,
        F.when(F.col(c) < lower, lower)
         .when(F.col(c) > upper, upper)
         .otherwise(F.col(c))
    )

spark.createDataFrame(outlier_rows, ["attribute", "q1", "q3", "lower", "upper"]).show(truncate=False)

imputer = Imputer(inputCols=NUMERIC_COLS, outputCols=[c + "_filled" for c in NUMERIC_COLS], strategy="median")
cleaned = imputer.fit(cleaned).transform(cleaned)

for c in NUMERIC_COLS:
    cleaned = cleaned.drop(c).withColumnRenamed(c + "_filled", c)

cleaned = cleaned.cache()

+---------+------------------+------------------+-------------------+------------------+
|attribute|q1                |q3                |lower              |upper             |
+---------+------------------+------------------+-------------------+------------------+
|u1_norm  |236.11            |241.0             |228.77500000000003 |248.33499999999998|
|u2_norm  |236.20000000000002|241.04            |228.94000000000005 |248.29999999999995|
|u3_norm  |236.67000000000002|241.0             |230.17500000000004 |247.49499999999998|
|i1_norm  |0.0               |10.4              |-15.600000000000001|26.0              |
|i2_norm  |0.176             |10.200000000000001|-14.860000000000001|25.236000000000004|
|i3_norm  |0.0               |10.200000000000001|-15.3              |25.5              |
+---------+------------------+------------------+-------------------+------------------+



## 9. Denne priebehy

Z 10-minutovych merani robim denne priebehy. Jeden cely den ma 144 merani. Pre kazdu velicinu pocitam priemer, minimum, maximum, sikmost a spicatost.

Den oznacim ako poruchovy, ak sa porucha vyskytla aspon v jednom merani daneho dna.

In [10]:
aggregations = []
for c in NUMERIC_COLS:
    aggregations += [
        F.avg(c).alias(c + "_avg"),
        F.max(c).alias(c + "_max"),
        F.min(c).alias(c + "_min"),
        F.skewness(c).alias(c + "_skewness"),
        F.kurtosis(c).alias(c + "_kurtosis"),
    ]

daily = (
    cleaned.withColumn("day", F.to_date("t_utc"))
    .groupBy("eic", "day")
    .agg(
        F.count("*").alias("n_samples"),
        F.max("target_binary").alias("target_binary"),
        F.array_sort(F.array_distinct(F.flatten(F.collect_list("fault_types")))).alias("daily_fault_types"),
        *aggregations
    )
    .where(F.col("n_samples") >= 144)
    .withColumn("target_multiclass", F.when(F.size("daily_fault_types") == 0, "0").otherwise(F.concat_ws("+", F.col("daily_fault_types"))))
    .drop("daily_fault_types")
    .cache()
)

print("Pocet dennych priebehov:", daily.count())
daily.groupBy("target_binary", "target_multiclass").count().orderBy("target_binary", "target_multiclass").show(50, truncate=False)

Pocet dennych priebehov: 52449
+-------------+-----------------+-----+
|target_binary|target_multiclass|count|
+-------------+-----------------+-----+
|0            |0                |21661|
|1            |1                |10793|
|1            |13               |230  |
|1            |2                |4071 |
|1            |3                |9190 |
|1            |4                |1870 |
|1            |5                |674  |
|1            |6                |484  |
|1            |7                |1549 |
|1            |8                |1354 |
|1            |9                |573  |
+-------------+-----------------+-----+



## 10. Train/test rozdelenie

In [11]:
# Rozdelenie robim stratifikovane podla binarneho ciela, aby boli v train aj test poruchove aj neporuchove dni.
w = Window.partitionBy("target_binary").orderBy(F.rand(SEED))
counts = daily.groupBy("target_binary").count().withColumnRenamed("count", "class_count")

split_df = (
    daily.join(counts, "target_binary")
    .withColumn("rn", F.row_number().over(w))
    .withColumn("is_train", F.col("rn") <= F.col("class_count") * 0.8)
)

train = split_df.where("is_train").drop("class_count", "rn", "is_train").cache()
test = split_df.where(~F.col("is_train")).drop("class_count", "rn", "is_train").cache()

print("Train:", train.count())
print("Test:", test.count())

Train: 41958
Test: 10491


## 11. Korelacie medzi atributmi

In [12]:
feature_cols = [
    c for c, dtype in daily.dtypes
    if c not in ["eic", "day", "target_binary", "target_multiclass"] and dtype in ["int", "bigint", "double"]
]

assembler_corr = VectorAssembler(inputCols=feature_cols, outputCol="features_corr", handleInvalid="skip")
corr_vector = Correlation.corr(assembler_corr.transform(train), "features_corr", "pearson").head()[0]
corr_array = corr_vector.toArray()

corr_rows = []
for i in range(len(feature_cols)):
    for j in range(i + 1, len(feature_cols)):
        corr_rows.append((feature_cols[i], feature_cols[j], float(corr_array[i, j]), abs(float(corr_array[i, j]))))

corr_pd = pd.DataFrame(corr_rows, columns=["attribute_1", "attribute_2", "correlation", "abs_correlation"])
corr_pd.sort_values("abs_correlation", ascending=False).head(20)

,attribute_1,attribute_2,correlation,abs_correlation
30,u1_norm_avg,u1_norm_max,0.945560,0.945560
165,u2_norm_avg,u2_norm_max,0.944957,0.944957
275,u3_norm_avg,u3_norm_max,0.941589,0.941589
31,u1_norm_avg,u1_norm_min,0.930309,0.930309
276,u3_norm_avg,u3_norm_min,0.925394,0.925394
166,u2_norm_avg,u2_norm_min,0.923884,0.923884
39,u1_norm_avg,u3_norm_avg,0.922467,0.922467
34,u1_norm_avg,u2_norm_avg,0.919073,0.919073
96,u1_norm_min,u3_norm_min,0.918898,0.918898
169,u2_norm_avg,u3_norm_avg,0.918258,0.918258


## 12. Vyber atributov podla Random Forest importance

In [13]:
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")
rf_for_importance = RandomForestClassifier(labelCol="target_binary", featuresCol="features", numTrees=80, seed=SEED)
rf_importance_model = rf_for_importance.fit(assembler.transform(train).select("target_binary", "features"))

importance_pd = pd.DataFrame({
    "attribute": feature_cols,
    "importance": rf_importance_model.featureImportances.toArray()
}).sort_values("importance", ascending=False)

selected_binary_features = importance_pd.head(20)["attribute"].tolist()
importance_pd.head(20)

,attribute,importance
21,i2_norm_avg,0.218843
17,i1_norm_max,0.147524
16,i1_norm_avg,0.129499
22,i2_norm_max,0.118236
26,i3_norm_avg,0.079767
27,i3_norm_max,0.051879
23,i2_norm_min,0.039282
28,i3_norm_min,0.032439
18,i1_norm_min,0.026075
7,u2_norm_max,0.018509


## 13. Funkcie na vyhodnotenie modelov

In [14]:
def calculate_metrics(predictions, label_col):
    # Metriky pocitam z kontingencnej tabulky, aby fungovalo binarne aj multiclass vyhodnotenie.
    rows = predictions.groupBy(label_col, "prediction").count().collect()
    labels = sorted({float(r[label_col]) for r in rows} | {float(r["prediction"]) for r in rows})
    matrix = {(float(r[label_col]), float(r["prediction"])): float(r["count"]) for r in rows}
    total = sum(matrix.values())

    correct = sum(matrix.get((x, x), 0.0) for x in labels)
    weighted_precision = 0.0
    weighted_recall = 0.0
    weighted_f1 = 0.0
    true_totals = {}
    pred_totals = {}

    for label in labels:
        tp = matrix.get((label, label), 0.0)
        true_total = sum(matrix.get((label, p), 0.0) for p in labels)
        pred_total = sum(matrix.get((t, label), 0.0) for t in labels)
        true_totals[label] = true_total
        pred_totals[label] = pred_total

        precision = tp / pred_total if pred_total else 0.0
        recall = tp / true_total if true_total else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

        weighted_precision += precision * true_total
        weighted_recall += recall * true_total
        weighted_f1 += f1 * true_total

    mcc_top = correct * total - sum(pred_totals[x] * true_totals[x] for x in labels)
    mcc_bottom_1 = total**2 - sum(pred_totals[x] ** 2 for x in labels)
    mcc_bottom_2 = total**2 - sum(true_totals[x] ** 2 for x in labels)
    mcc_bottom = math.sqrt(mcc_bottom_1 * mcc_bottom_2) if mcc_bottom_1 > 0 and mcc_bottom_2 > 0 else 0

    return {
        "accuracy": correct / total,
        "precision": weighted_precision / total,
        "recall": weighted_recall / total,
        "f1": weighted_f1 / total,
        "mcc": mcc_top / mcc_bottom if mcc_bottom else 0.0,
    }


def train_and_evaluate(model_name, model, train_df, test_df, features, label_col):
    pipeline = Pipeline(stages=[
        VectorAssembler(inputCols=features, outputCol="features", handleInvalid="skip"),
        model
    ])

    fitted = pipeline.fit(train_df)
    predictions = fitted.transform(test_df).cache()
    metrics = calculate_metrics(predictions, label_col)
    confusion = predictions.groupBy(label_col, "prediction").count().orderBy(label_col, "prediction")

    return {"model": model_name, **metrics}, confusion

## 14. Binarna klasifikacia - lubovolna porucha

In [15]:
binary_models = [
    ("logistic_regression", LogisticRegression(labelCol="target_binary", featuresCol="features", maxIter=50)),
    ("random_forest", RandomForestClassifier(labelCol="target_binary", featuresCol="features", numTrees=100, maxDepth=8, seed=SEED)),
    ("gradient_boosted_trees", GBTClassifier(labelCol="target_binary", featuresCol="features", maxIter=50, maxDepth=5, seed=SEED)),
]

binary_results = []
binary_confusions = {}

for name, model in binary_models:
    metrics, confusion = train_and_evaluate(name, model, train, test, selected_binary_features, "target_binary")
    binary_results.append(metrics)
    binary_confusions[name] = confusion

binary_results_pd = pd.DataFrame(binary_results).sort_values(["f1", "mcc"], ascending=False)
binary_results_pd

,model,accuracy,precision,recall,f1,mcc
2,gradient_boosted_trees,0.977917,0.978293,0.977917,0.977923,0.956181
1,random_forest,0.972686,0.973725,0.972686,0.972693,0.946393
0,logistic_regression,0.830161,0.834247,0.830161,0.829218,0.663344


In [16]:
best_binary_model = binary_results_pd.iloc[0]["model"]
print("Najlepsi binarny model:", best_binary_model)
binary_confusions[best_binary_model].show()

Najlepsi binarny model: gradient_boosted_trees
+-------------+----------+-----+
|target_binary|prediction|count|
+-------------+----------+-----+
|            0|       0.0| 3298|
|            0|       1.0|   29|
|            1|       0.0|  123|
|            1|       1.0| 3433|
+-------------+----------+-----+



## 15. Multiclass klasifikacia - typ poruchy alebo kombinacia poruch

In [17]:
# Typ poruchy je text, napr. "0", "1", "4" alebo kombinacia "1+4". Pre Spark ML ho prevediem na index.
indexer = StringIndexer(inputCol="target_multiclass", outputCol="target_multiclass_index", handleInvalid="keep")
indexer_model = indexer.fit(daily)
daily_multi = indexer_model.transform(daily).cache()

split_multi = (
    daily_multi.join(counts, "target_binary")
    .withColumn("rn", F.row_number().over(w))
    .withColumn("is_train", F.col("rn") <= F.col("class_count") * 0.8)
)

train_multi = split_multi.where("is_train").drop("class_count", "rn", "is_train").cache()
test_multi = split_multi.where(~F.col("is_train")).drop("class_count", "rn", "is_train").cache()

multi_feature_cols = [
    c for c, dtype in daily_multi.dtypes
    if c not in ["eic", "day", "target_binary", "target_multiclass", "target_multiclass_index"] and dtype in ["int", "bigint", "double"]
]

assembler_multi = VectorAssembler(inputCols=multi_feature_cols, outputCol="features", handleInvalid="skip")
rf_multi_importance = RandomForestClassifier(labelCol="target_multiclass_index", featuresCol="features", numTrees=80, seed=SEED)
rf_multi_model = rf_multi_importance.fit(assembler_multi.transform(train_multi).select("target_multiclass_index", "features"))

multi_importance_pd = pd.DataFrame({
    "attribute": multi_feature_cols,
    "importance": rf_multi_model.featureImportances.toArray()
}).sort_values("importance", ascending=False)

selected_multi_features = multi_importance_pd.head(20)["attribute"].tolist()
multi_importance_pd.head(20)

,attribute,importance
21,i2_norm_avg,0.167601
17,i1_norm_max,0.146584
16,i1_norm_avg,0.121907
22,i2_norm_max,0.118665
27,i3_norm_max,0.095347
26,i3_norm_avg,0.057477
23,i2_norm_min,0.039802
28,i3_norm_min,0.032753
6,u2_norm_avg,0.025904
7,u2_norm_max,0.024047


In [18]:
multiclass_models = [
    ("logistic_regression", LogisticRegression(labelCol="target_multiclass_index", featuresCol="features", maxIter=50, family="multinomial")),
    ("random_forest", RandomForestClassifier(labelCol="target_multiclass_index", featuresCol="features", numTrees=100, maxDepth=8, seed=SEED)),
    ("decision_tree", DecisionTreeClassifier(labelCol="target_multiclass_index", featuresCol="features", maxDepth=8, seed=SEED)),
]

multiclass_results = []
multiclass_confusions = {}

for name, model in multiclass_models:
    metrics, confusion = train_and_evaluate(name, model, train_multi, test_multi, selected_multi_features, "target_multiclass_index")
    multiclass_results.append(metrics)
    multiclass_confusions[name] = confusion

multiclass_results_pd = pd.DataFrame(multiclass_results).sort_values(["f1", "mcc"], ascending=False)
multiclass_results_pd

,model,accuracy,precision,recall,f1,mcc
1,random_forest,0.952375,0.954071,0.952375,0.949919,0.931252
2,decision_tree,0.951715,0.952577,0.951715,0.949709,0.930359
0,logistic_regression,0.878100,0.870988,0.878100,0.867472,0.825069


In [19]:
best_multiclass_model = multiclass_results_pd.iloc[0]["model"]
print("Najlepsi multiclass model:", best_multiclass_model)
multiclass_confusions[best_multiclass_model].show(100)

Najlepsi multiclass model: random_forest
+-----------------------+----------+-----+
|target_multiclass_index|prediction|count|
+-----------------------+----------+-----+
|                    0.0|       0.0| 3599|
|                    0.0|       1.0|    2|
|                    0.0|       2.0|   38|
|                    0.0|       3.0|    6|
|                    0.0|       4.0|    2|
|                    0.0|       7.0|    2|
|                    1.0|       0.0|   26|
|                    1.0|       1.0|  417|
|                    1.0|       3.0|    2|
|                    2.0|       0.0|   46|
|                    2.0|       2.0| 1723|
|                    3.0|       0.0|    7|
|                    3.0|       3.0|  762|
|                    4.0|       0.0|   12|
|                    4.0|       4.0|  213|
|                    5.0|       0.0|   19|
|                    5.0|       1.0|    5|
|                    5.0|       5.0|  109|
|                    5.0|       9.0|    1|
|            

## 16. Zaver

V notebooku boli splnene vsetky kroky zadania. Ako cielove atributy som pouzil:

- `target_binary` - ci je v danom merani alebo dni aspon jedna porucha,
- `target_multiclass` - konkretny typ poruchy alebo kombinacia viacerych sucasnych poruch.

Najlepsi model vyberam podla F1 a MCC, pretoze pri nevyvazenych triedach su vhodnejsie ako samotna accuracy.

In [20]:
print("Najlepsi binarny model:", best_binary_model)
display(binary_results_pd)

print("Najlepsi multiclass model:", best_multiclass_model)
display(multiclass_results_pd)

spark.stop()

Najlepsi binarny model: gradient_boosted_trees


,model,accuracy,precision,recall,f1,mcc
2,gradient_boosted_trees,0.977917,0.978293,0.977917,0.977923,0.956181
1,random_forest,0.972686,0.973725,0.972686,0.972693,0.946393
0,logistic_regression,0.830161,0.834247,0.830161,0.829218,0.663344


Najlepsi multiclass model: random_forest


,model,accuracy,precision,recall,f1,mcc
1,random_forest,0.952375,0.954071,0.952375,0.949919,0.931252
2,decision_tree,0.951715,0.952577,0.951715,0.949709,0.930359
0,logistic_regression,0.878100,0.870988,0.878100,0.867472,0.825069
